In [1]:
%%writefile pipeline.py
"""Shared data loading and preprocessing for the Task 3-5 notebooks."""
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

CACHE_DIR = Path("cache")
IMG_SIZE = 64            # must match the value used in Task 2
SEED = 42


def load_cached_data():
    """Reload the arrays, labels and train/validation split that the Task 2 notebook saved."""
    split = np.load(CACHE_DIR / "split.npz")
    y_all = torch.from_numpy(split["y_all"]).long()
    class_to_idx = json.load(open(CACHE_DIR / "class_to_idx.json"))
    classes = sorted(class_to_idx, key=class_to_idx.get)          # class names in index order

    X_all = torch.from_numpy(np.load(CACHE_DIR / f"train_{IMG_SIZE}_{len(y_all)}.npy"))
    X_all = X_all.permute(0, 3, 1, 2).contiguous()                # (N,H,W,C) -> (N,C,H,W)
    test_file = sorted(CACHE_DIR.glob(f"test_{IMG_SIZE}_*.npy"))[0]
    X_test = torch.from_numpy(np.load(test_file)).permute(0, 3, 1, 2).contiguous()

    idx_tr, idx_va = split["idx_tr"], split["idx_va"]
    stats = {"train": (torch.from_numpy(split["mean"]), torch.from_numpy(split["std"])),
             "imagenet": (torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225]))}
    return dict(X_tr=X_all[idx_tr], y_tr=y_all[idx_tr], X_va=X_all[idx_va], y_va=y_all[idx_va],
                X_test=X_test, classes=classes, stats=stats)


def augment(x, max_shift=4):
    """x: float batch in [0,1], shape (B,3,H,W). Every image gets its own random changes."""
    B, C, H, W = x.shape
    flip = torch.rand(B) < 0.5                                    # 1. horizontal flip
    x = torch.where(flip[:, None, None, None], x.flip(3), x)

    p = max_shift                                                 # 2. random shift
    xp = F.pad(x, (p, p, p, p), mode="reflect")
    oy = torch.randint(0, 2 * p + 1, (B,)); ox = torch.randint(0, 2 * p + 1, (B,))
    rows = oy[:, None] + torch.arange(H)
    cols = ox[:, None] + torch.arange(W)
    x = xp[torch.arange(B)[:, None, None], :, rows[:, :, None], cols[:, None, :]].permute(0, 3, 1, 2)

    bright = (torch.rand(B, 1, 1, 1) - 0.5) * 0.4                 # 3. brightness / contrast
    contrast = 1 + (torch.rand(B, 1, 1, 1) - 0.5) * 0.4
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    return ((x - m) * contrast + m + bright).clamp(0, 1)


def prepare(xb_uint8, train, stats, use_aug=True):
    """uint8 batch -> float in [0,1] -> augment (training only) -> normalise with stats=(mean, std)."""
    x = xb_uint8.float() / 255
    if train and use_aug:
        x = augment(x)
    mean, std = stats
    return ((x - mean[None, :, None, None]) / std[None, :, None, None]).contiguous()

Overwriting pipeline.py


In [2]:
%%writefile model_defs.py
"""Model definitions for the Task 3-5 notebooks."""
from functools import partial

import torch.nn as nn
from torchvision import models

NUM_CLASSES = 8


def conv_block(cin, cout):
    """3x3 convolution -> batch normalisation -> ReLU."""
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class CustomCNN(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, widths=(32, 64, 128, 256), dropout=0.3):
        super().__init__()
        layers, cin = [], 3
        for i, w in enumerate(widths):
            layers.append(conv_block(cin, w))
            if i >= 2:                                   # deeper blocks get a second conv
                layers.append(conv_block(w, w))
            layers.append(nn.MaxPool2d(2))               # halve the height and width
            cin = w
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(cin, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def build_custom_cnn(pretrained=True):                   # `pretrained` is ignored (trained from scratch)
    return CustomCNN()


def build_resnet18(pretrained=True, freeze_early=False):
    m, loaded = None, False
    if pretrained:
        try:
            m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)   # downloads once (needs internet)
            loaded = True
        except Exception as e:
            print("WARNING: could not load pretrained weights, using random init instead:", e)
    if m is None:
        m = models.resnet18(weights=None)
    if freeze_early:
        if pretrained and not loaded:
            print("WARNING: early layers are frozen but the weights are NOT pretrained")
        for part in (m.conv1, m.bn1, m.layer1, m.layer2):   # freeze the early, high-resolution layers
            for p in part.parameters():
                p.requires_grad = False
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, NUM_CLASSES))
    return m


MODEL_CONFIGS = {
    "custom_cnn":       dict(build=build_custom_cnn, stats="train",
                             lr=2e-3, weight_decay=5e-4, epochs=12),
    "resnet18_pt":      dict(build=build_resnet18, stats="imagenet",
                             lr=1e-3, weight_decay=1e-4, epochs=6),
    "resnet18_partial": dict(build=partial(build_resnet18, freeze_early=True), stats="imagenet",
                             lr=1e-3, weight_decay=1e-4, epochs=6),
}

Overwriting model_defs.py


In [5]:
import os, time
import numpy as np
import torch # Might need pip install torchvision
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pipeline as pl
import model_defs as md

torch.manual_seed(pl.SEED)
data = pl.load_cached_data()
X_tr, y_tr, X_va, y_va = data["X_tr"], data["y_tr"], data["X_va"], data["y_va"]
classes, STATS = data["classes"], data["stats"]
print("train:", tuple(X_tr.shape), X_tr.dtype, "| validation:", tuple(X_va.shape), "| test:", tuple(data["X_test"].shape))
print("torch threads:", torch.get_num_threads(), "| logical cores:", os.cpu_count())

g = torch.Generator(); g.manual_seed(pl.SEED)
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True, generator=g)
print("batches per epoch:", len(train_loader))

for name, cfg in md.MODEL_CONFIGS.items():
    m = cfg["build"](pretrained=False)                   # no download needed just to inspect the shape
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    out = m(torch.zeros(2, 3, pl.IMG_SIZE, pl.IMG_SIZE))
    print(f"{name:17s} parameters {total:>10,} | trainable {trainable:>10,} | output {tuple(out.shape)}")

train: (50916, 3, 64, 64) torch.uint8 | validation: (12729, 3, 64, 64) | test: (29443, 3, 64, 64)
torch threads: 10 | logical cores: 12
batches per epoch: 398
custom_cnn        parameters  1,129,000 | trainable  1,129,000 | output (2, 8)
resnet18_pt       parameters 11,180,616 | trainable 11,180,616 | output (2, 8)
resnet18_partial  parameters 11,180,616 | trainable 10,497,544 | output (2, 8)


In [6]:
def seconds_per_step(name, steps=8):
    cfg = md.MODEL_CONFIGS[name]
    model = cfg["build"](); model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-3)
    it = iter(train_loader); times = []
    for _ in range(steps + 1):
        xb, yb = next(it)
        t = time.time()
        loss = F.cross_entropy(model(pl.prepare(xb, True, STATS[cfg["stats"]])), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        times.append(time.time() - t)
    return float(np.mean(times[1:]))                     # skip the first (warm-up) step

for name in md.MODEL_CONFIGS:
    s = seconds_per_step(name)
    print(f"{name:17s} {s:.2f} s/step -> about {s * len(train_loader) / 60:.1f} min/epoch")

# does the number of CPU threads matter? (your CPU mixes fast and slow cores)
default_threads = torch.get_num_threads()
for t in sorted({4, 6, 8, default_threads, os.cpu_count()}):
    if t <= os.cpu_count():
        torch.set_num_threads(t)
        print(f"{t:2d} threads: {seconds_per_step('resnet18_pt', steps=5):.2f} s/step")
torch.set_num_threads(default_threads)

custom_cnn        0.84 s/step -> about 5.6 min/epoch
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\devme/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 48.2MB/s]


resnet18_pt       2.45 s/step -> about 16.2 min/epoch
resnet18_partial  1.56 s/step -> about 10.4 min/epoch
 4 threads: 2.60 s/step
 6 threads: 2.16 s/step
 8 threads: 1.88 s/step
10 threads: 1.68 s/step
12 threads: 2.12 s/step
